In [1]:
import base64
import datasets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import warnings

warnings.simplefilter(action = 'ignore', category = FutureWarning)

# Preprocessing Functions

In [3]:
def decode_base64(item):
    return base64.b64decode(item.encode()).decode('utf-8')

In [4]:
def decode_human_ai_hash(df):
    df = df.drop(columns=[col for col in df.columns if 'Unnamed' in col])
    df = df.applymap(lambda x: x.replace('\xa0', '').strip() if isinstance(x, str) else x)
    df['hash'] = df['hash'].apply(decode_base64)
    df['REQID_ex'] = df['hash'].str.split('__________').str[0]
    df['author'] = df['hash'].str.split('__________').str[-1]
    df.drop(columns = ['hash'], inplace = True)

    nan_summary = df.isnull().sum()
    print("Number of NaN values in each column:\n", nan_summary)
    for column in df.columns:
        if df[column].isnull().sum() > 0:
            majority_value = df[column].mode()[0]
            df[column].fillna(majority_value, inplace = True)
    return df

In [5]:
participants = ['p1', 'p2', 'p3', 'p4']

human_assessed_requirements = [pd.read_excel(f'./{i}_human_evaluation/tasks_b_and_c/task_b.xlsx') for i in participants]
human_assessed_requirements = [decode_human_ai_hash(df) for df in human_assessed_requirements]

Number of NaN values in each column:
 requirement                                                                                                        0
Based on the style and content of the requirement, do you believe it was written by a human or generated by AI?    0
This requirement is well-structured according to the ISO-29248 recommended syntax.                                 0
The use of signaling keywords to indicate the presence of a requirement is appropriate based on ISO-29148.         0
REQID_ex                                                                                                           0
author                                                                                                             0
dtype: int64
Number of NaN values in each column:
 requirement                                                                                                        0
Based on the style and content of the requirement, do you believe it was written by a human 

In [6]:
ai_human_label2id = {'Zephyr': 0, 'ReqBrain': 1}
likert_label2id = {'Strongly Disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly Agree': 5}

likert_id2label = {v: k for k, v in zip(likert_label2id.keys(), likert_label2id.values())}
ai_human_id2label = {v: k for k, v in zip(ai_human_label2id.keys(), ai_human_label2id.values())}

In [7]:
human_ai, syntax_quality, keyword_quality = human_assessed_requirements[0].columns[1:-2]

In [8]:
# Reproduce the shuffle used when the evaluation files were created
pair_reconstruction = pd.DataFrame({'original_position': range(68)})
pair_reconstruction = pair_reconstruction.sample(frac = 1, random_state = 42).reset_index(drop = True)
pair_ids = (pair_reconstruction['original_position'] % 34).to_numpy()

for df in human_assessed_requirements:
    assert len(df) == 68
    df['pair_id'] = pair_ids

In [9]:
for df in human_assessed_requirements:
    df['pair_id'] = pair_ids

for df in human_assessed_requirements:
    df[syntax_quality] = df[syntax_quality].map(likert_label2id)
    df[keyword_quality] = df[keyword_quality].map(likert_label2id)

human_assessed_requirements_concated = pd.concat(human_assessed_requirements, axis = 0, ignore_index = True)

def aggregate_binary_ratings(ratings):
    human_votes = (ratings == 'HUMAN').sum()
    return 'HUMAN' if human_votes >= 3 else 'AI'

ratings_per_requirement = human_assessed_requirements_concated.groupby(['pair_id', 'requirement', 'REQID_ex', 'author']).size()

human_assessed_requirements_concated = (
    human_assessed_requirements_concated
    .groupby(['pair_id', 'requirement', 'REQID_ex', 'author'], as_index = False)
    .agg({human_ai: aggregate_binary_ratings, syntax_quality: 'median', keyword_quality: 'median'})
)

human_assessed_requirements_concated['author'] = human_assessed_requirements_concated['author'].replace({'AI': 'ReqBrain', 'UAI': 'Zephyr'})

pair_validation = human_assessed_requirements_concated.groupby(['pair_id', 'author']).size().unstack(fill_value = 0)

In [10]:
print('Number of aggregated requirements:', len(human_assessed_requirements_concated))
print('Number of matched pairs:', len(pair_validation))

print('\nRequirements per source:')
print(human_assessed_requirements_concated['author'].value_counts())

print('\nAggregated binary results:')
print(human_assessed_requirements_concated[human_ai].value_counts())

Number of aggregated requirements: 68
Number of matched pairs: 34

Requirements per source:
author
Zephyr      34
ReqBrain    34
Name: count, dtype: int64

Aggregated binary results:
Based on the style and content of the requirement, do you believe it was written by a human or generated by AI?
AI       56
HUMAN    12
Name: count, dtype: int64


In [11]:
# mapping labels to id mapping
human_assessed_requirements_concated['author_numeric'] = human_assessed_requirements_concated['author'].map(ai_human_label2id)

## **Variable Name:** Perceived Authorship$_{(PA)}$  
**Variable Description:** Based on the style and content of the requirement, do you believe it was written by a human or generated by AI?


## **Hypotheses:**

- **H$_{0,1}$:** Fine-tuning does not increase perceived authenticity, as ReqBrain-generated requirements are not identified as human-written in a greater proportion than their matched-pair counterparts from its untuned baseline model.
- **H$_{a,1}$:** Fine-tuning increases perceived authenticity, as ReqBrain-generated requirements are identified as human-written in a greater proportion than their matched-pair counterparts from its untuned baseline model.

# Code

In [12]:
def counts_2ind_samples(sample1, sample2):
    counts_sample1 = np.unique(sample1, return_counts = True)[-1][-1]
    counts_sample2 = np.unique(sample2, return_counts = True)[-1][-1]
    
    nominator = [counts_sample1, counts_sample2][::-1]
    denominator = [len(sample1)] * 2
    
    return np.array(nominator), np.array(denominator)

In [13]:
def calculate_ci_from_odds_ratio(odds_ratio, table):
    log_or = np.log(odds_ratio)
    se_log_or = np.sqrt(1 / table[0][0] + 1 / table[0][1] + 1 / table[1][0] + 1 / table[1][1])
    ci_lower_log = log_or - 1.96 * se_log_or
    ci_upper_log = log_or + 1.96 * se_log_or
    ci_lower = np.exp(ci_lower_log)
    ci_upper = np.exp(ci_upper_log)
    return ci_lower, ci_upper,

In [14]:
zephyr = human_assessed_requirements_concated[human_assessed_requirements_concated['author'] == 'Zephyr']
ReqBrain = human_assessed_requirements_concated[human_assessed_requirements_concated['author'] == 'ReqBrain']
raw_ratings = np.array([human_assessed_requirements[p].loc[:, human_ai].to_numpy() for p in range(len(participants))], dtype = 'object').T

In [15]:
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

def fleiss_kappa_ci(raw_ratings, confidence = 0.95, n_resamples = 2000, seed = 42):
    counts_local, _ = aggregate_raters(raw_ratings)
    n_items = counts_local.shape[0]
    kappa_samples = []

    rng = np.random.RandomState(seed)
    for i in range(n_resamples):
        indices = rng.choice(n_items, size = n_items, replace = True)
        bootstrap_sample = counts_local[indices]
        try:
            kappa_value = fleiss_kappa(bootstrap_sample)
            if np.isfinite(kappa_value):
                kappa_samples.append(kappa_value)
        except (AssertionError, ValueError):
            continue
    if len(kappa_samples) < 100:
        print(f"Warning: Only {len(kappa_samples)} valid bootstrap samples")
        return np.nan, np.nan

    alpha = (1 - confidence) / 2
    lower = np.percentile(kappa_samples, alpha * 100)
    upper = np.percentile(kappa_samples, (1 - alpha) * 100)

    return float(lower), float(upper)

In [16]:
from scipy.stats import binomtest

paired = ReqBrain[['pair_id', human_ai]].merge(zephyr[['pair_id', human_ai]], on = 'pair_id', suffixes = ('_ReqBrain', '_Zephyr'), validate = 'one_to_one')

assert len(paired) == 34

table = pd.crosstab(paired[f'{human_ai}_ReqBrain'], paired[f'{human_ai}_Zephyr']).reindex(index = ['AI', 'HUMAN'], columns = ['AI', 'HUMAN'], fill_value = 0)

b = table.loc['AI', 'HUMAN']
c = table.loc['HUMAN', 'AI']
discordant_pairs = b + c
mcnemar_statistic = min(b, c)

'''# This binomial test is the exact form of McNemar's test.
# Exact McNemar conditions on the total number of discordant pairs (b + c).
# Under H0, each discordant pair has probability 0.5 of falling in either
# direction, so c ~ Binomial(b + c, 0.5).
# Therefore, binomtest(c, b + c, 0.5) implements exact McNemar.
# alternative='greater' tests whether ReqBrain is classified as HUMAN
# more often than Zephyr.'''
result = binomtest(c, n = b + c, p = 0.5, alternative = 'greater')

p_value_0 = result.pvalue
odds_ratio = c / b if b > 0 else np.inf

# Exact confidence interval for matched odds ratio
proportion_ci = binomtest(c, n = b + c).proportion_ci(confidence_level = 0.95, method = 'exact')

ci_lower = (proportion_ci.low / (1 - proportion_ci.low) if proportion_ci.low > 0 else 0)
ci_upper = (proportion_ci.high / (1 - proportion_ci.high) if proportion_ci.high < 1 else np.inf)

reqbrain_human = (ReqBrain[human_ai] == 'HUMAN').sum()
zephyr_human = (zephyr[human_ai] == 'HUMAN').sum()


counts, cats = aggregate_raters(raw_ratings)
fleiss_statistics = fleiss_kappa(counts)
kappa_low, kapp_high = fleiss_kappa_ci(raw_ratings)
print("Categories:", list(cats))
print(f"Fleiss' Kappa: {fleiss_statistics:.4f} | 95% CI: [{kappa_low:.4f}, {kapp_high:.4f}]\n")


print(f'P-Value: {p_value_0:.5f}')
print(f'Odds Ratio: {odds_ratio:.3f}')
print(f'Total discordant pairs: {discordant_pairs}')
print(f'Exact McNemar statistic: {mcnemar_statistic}')


print('\n')
print(f'Odds Ratio: Effect Size 95% CI Lower Bound: {ci_lower:.2f}')
print(f'Odds Ratio: Effect Size 95% CI Upper Bound: {ci_upper:.2f}')

print('\n')
print('Sample-1 (ReqBrain)')
print(f'Sample size: {len(ReqBrain)}')
print(f'Classified as human count (success count): {reqbrain_human}')
print(f'Classified as AI count (failure count): {len(ReqBrain) - reqbrain_human}')
print(f'Proportion: {reqbrain_human / len(ReqBrain)}')

print('\n')
print('Sample-2 (Zephyr)')
print(f'Sample size: {len(zephyr)}')
print(f'Classified as human count (success count): {zephyr_human}')
print(f'Classified as AI count (failure count): {len(zephyr) - zephyr_human}')
print(f'Proportion: {zephyr_human / len(zephyr)}')

Categories: ['AI', 'HUMAN']
Fleiss' Kappa: 0.2814 | 95% CI: [0.1534, 0.4003]

P-Value: 0.00391
Odds Ratio: inf
Total discordant pairs: 8
Exact McNemar statistic: 0


Odds Ratio: Effect Size 95% CI Lower Bound: 1.71
Odds Ratio: Effect Size 95% CI Upper Bound: inf


Sample-1 (ReqBrain)
Sample size: 34
Classified as human count (success count): 10
Classified as AI count (failure count): 24
Proportion: 0.29411764705882354


Sample-2 (Zephyr)
Sample size: 34
Classified as human count (success count): 2
Classified as AI count (failure count): 32
Proportion: 0.058823529411764705


## **Variable Name:** Written Syntax Compliance$_{(WSC)}$ 
**Variable Description:** This requirement is well-structured according to the ISO-29248 recommended syntax.


## **Hypotheses:**
- **$H_{0,3}$:** ReqBrain-generated requirements do not show greater adherence to ISO~29148 syntax than their matched-pair counterparts from ReqBrain's untuned baseline.

- **$H_{a,3}$:** ReqBrain-generated requirements show greater adherence to ISO~29148 syntax than their matched-pair counterparts from ReqBrain's untuned baseline.

# Code 

In [17]:
# Calculating basic statistics
def calculate_statistics(sample):
    n = len(sample)
    mean = np.mean(sample)
    median = np.median(sample)
    std_dev = np.std(sample, ddof = 1)

    print(f'n {n}')
    print(f'Mean {mean:.3f}')
    print(f'Median {median:.3f}')
    print(f'Standard Deviation {std_dev:.3f}')

In [18]:
from scipy.stats import wilcoxon, rankdata, bootstrap

# Calculating matched-pairs rank-biserial effect size
def rank_biserial_correlation(sample_1, sample_2):
    differences = np.asarray(sample_1) - np.asarray(sample_2)

    # Wilcoxon excludes zero differences
    differences = differences[differences != 0]

    if len(differences) == 0:
        return 0.0

    ranks = rankdata(np.abs(differences))

    positive_ranks = ranks[differences > 0].sum()
    negative_ranks = ranks[differences < 0].sum()

    return (positive_ranks - negative_ranks) / (positive_ranks + negative_ranks)

In [19]:
# Calculating paired bootstrap CI
def bootstrapped_ci(sample_1, sample_2, n_samples = 1000, ci = .95):
    samples = (np.asarray(sample_1),np.asarray(sample_2))
    result = bootstrap(samples, rank_biserial_correlation, n_resamples = n_samples,confidence_level = ci, method = 'percentile', vectorized = False, paired = True, random_state = 42)            
    ci_lower, ci_upper = result.confidence_interval
    return ci, ci_lower, ci_upper

In [20]:
# Run all statistics
def stats_report(sample_1, sample_2):
    sample_1 = np.asarray(sample_1)
    sample_2 = np.asarray(sample_2)

    assert len(sample_1) == len(sample_2), \
        "The paired samples must have equal lengths."

    print("Sample 1: ReqBrain Authored\n")
    calculate_statistics(sample_1)

    print('\n')
    print("Sample 2: Untrained Zephyr Authored\n")
    calculate_statistics(sample_2)

    print('\n')
    w_stat, p_value = wilcoxon(sample_1, sample_2, alternative = 'greater', zero_method = 'wilcox', method = 'auto')

    print(f"p_value: {p_value:.5f}")
    print("Wilcoxon Signed-Rank Statistics:", w_stat)

    print('\n')
    
    effect_size = rank_biserial_correlation(sample_1, sample_2)
    print(f"Matched rank-biserial correlation: {effect_size:.3f}")

    ci, ci_lower, ci_upper = bootstrapped_ci(sample_1, sample_2)
    print(f"{int(ci * 100)}% Confidence Interval for matched rank-biserial correlation: ({ci_lower:.5f}, {ci_upper:.5f})")

    return p_value

In [21]:
reqbrain_sorted = ReqBrain.sort_values('pair_id')
zephyr_sorted = zephyr.sort_values('pair_id')

p_value_1 = stats_report(reqbrain_sorted[syntax_quality], zephyr_sorted[syntax_quality])

Sample 1: ReqBrain Authored

n 34
Mean 3.779
Median 4.000
Standard Deviation 1.088


Sample 2: Untrained Zephyr Authored

n 34
Mean 2.118
Median 2.000
Standard Deviation 0.985


p_value: 0.00000
Wilcoxon Signed-Rank Statistics: 544.0


Matched rank-biserial correlation: 0.939
95% Confidence Interval for matched rank-biserial correlation: (0.81815, 0.99328)


## **Variable Name:** Signaling Keywords Compliance$_{(SKC)}$  
**Variable Description:** The use of signaling keywords to indicate the presence of a requirement is appropriate based on ISO-29148.

## **Hypotheses:**
- **$H_{0,4}$:** ReqBrain-generated requirements do not show greater adherence to ISO~29148 signaling keywords than their matched-pair counterparts from ReqBrain's untuned baseline.

- **$H_{a,4}$:** ReqBrain-generated requirements show greater adherence to ISO~29148 signaling keywords than their matched-pair counterparts from ReqBrain's untuned baseline.

# Code

In [22]:
p_value_2 = stats_report(reqbrain_sorted[keyword_quality], zephyr_sorted[keyword_quality])

Sample 1: ReqBrain Authored

n 34
Mean 3.971
Median 4.500
Standard Deviation 1.051


Sample 2: Untrained Zephyr Authored

n 34
Mean 2.676
Median 2.500
Standard Deviation 0.976


p_value: 0.00002
Wilcoxon Signed-Rank Statistics: 485.0


Matched rank-biserial correlation: 0.837
95% Confidence Interval for matched rank-biserial correlation: (0.63622, 0.95630)


# **Holm-Bonferroni Correction**

In [23]:
from statsmodels.stats.multitest import multipletests

# Example list of p-values
pvals = [p_value_0, p_value_1, p_value_2]

# Perform Holm-Bonferroni correction
reject, pvals_corrected, _, _ = multipletests(pvals, alpha = 0.05, method = 'holm')

print("Original p-values:", [f"{i:.5f}" for i in pvals])
print("Adjusted p-values:", [f"{i:.5f}" for i in pvals_corrected])
print("Reject null hypothesis:", reject)

Original p-values: ['0.00391', '0.00000', '0.00002']
Adjusted p-values: ['0.00391', '0.00000', '0.00003']
Reject null hypothesis: [ True  True  True]
